# Objects

In Python, everything is an object. 

In [1]:
a = 1
def hi(): ...
class Dog: ...

print(isinstance(a, object))
print(isinstance(hi, object))
print(isinstance(Dog, object))

True
True
True


Even functions and classes are objects.

**But what is an object?**

Conceptually, an object is something with an identity, a type, and a value

- Identity makes the object distinct from every other object
- Type defines the object's behaviour
- Value holds the object's data

CPython implements this as a C struct. Every object’s struct starts with the same header consisting of

- A reference count (for memory management)
- A pointer to its type

it then adds its own fields to store its value. Note that identity isn't explicitly stored anywhere - CPython uses the struct's memory address as the identity.

For example,

- `5` is an object that contains a reference count, a pointer to the `int` type, and its value, which is the number 5 in binary plus its sign and length.
- `int` is an object that contains a reference count, a pointer to the `type` type, and its value, which is a bunch of stuff that defines how int instances behave.

In [2]:
x = 5

# identity, type, value
print(id(x), type(x), x)

# the type chain: 5 → int → type → type
print(type(5), type(int), type(type))

191710153365824 <class 'int'> 5
<class 'int'> <class 'type'> <class 'type'>


# Namespaces

In Python, every "variable" is a name, which is bound to an object within a namespace.

Conceptually, a namespace is a mapping from names to objects. CPython implements them as dict attributes of objects (except for function locals), for example:

- The built-in namespace is stored in `builtins.__dict__`
- The module namespace is stored in `sys.modules[__name__].__dict__` or `globals()`
- The class namespace is stored in `Class.__dict__`
- The instance namespace is stored in `obj.__dict__`
- The function local level in internal slots for better performance

In [3]:
import builtins

x = "global x"

class Parent:
    p = "from Parent"

class Child(Parent):
    c = "from Child"
    def __init__(self):
        self.i = "from instance"

obj = Child()
obj.hi = "hi"

print("'len' in builtins.__dict__: ", "len" in builtins.__dict__)
print("'x'   in globals():         ", "x" in globals())
print("'p'   in Parent.__dict__:   ", "p" in Parent.__dict__)
print("'p'   in Child.__dict__:    ", "p" in Child.__dict__)
print("'c'   in Child.__dict__:    ", "c" in Child.__dict__)
print("'c'   in obj.__dict__:      ", "c" in obj.__dict__)
print("obj.__dict__:               ", obj.__dict__)

'len' in builtins.__dict__:  True
'x'   in globals():          True
'p'   in Parent.__dict__:    True
'p'   in Child.__dict__:     False
'c'   in Child.__dict__:     True
'c'   in obj.__dict__:       False
obj.__dict__:                {'i': 'from instance', 'hi': 'hi'}


When you use a name, CPython follows a fixed lookup order

For bare names (`x`), it uses the LEGB rule

- Local: the local namespace of the current function calls
- Enclosing: the local namespace of any outer function calls
- Global: the module namespace
- Built-in: the built-in namespace

For dotted names (`obj.x`), it searches through

- Instance namespace
- Class namespace
- Parent classes' namespaces

**But what does 'bound' actually mean?**

Let's build a mental picture compared to other types of variables:

- In C, a plain variable holds a value
- In C, a pointer holds a memory address
- In C++, a reference is like a C pointer that automatically dereferences, but it can never be re-pointed
- In Python, a name is like a C++ reference, but it can be re-pointed via the assignment operator.

# Generator vs Coroutine

When `def` or `async def` runs, it creates a function object. When you call them, however, it behaves differently dependnig on the kind of function. When you call a function defined via `async def`, it creates a coroutine object which holds a frame object so it can be paused and resumed. When you call a function defined via `def` and `yield`, it creates a generator object which can similarly be paused and resumed. Calling a plain function returns a `None` object.

Generators and coroutine are both functions that can be paused and resumed. The main difference comes down to intent.

Generators were designed for streaming, or lazily loading, data. For example, a generator can yield elements of a large sequence one at a time without loading everything into memory at once.

Plain coroutines are basically generators, executing synchronously in a nested call stack. However, if you wrap them in an `asyncio.Task` object (more below), you can interleave tasks driven by the event loop.

In [10]:
print("--- Generator function ---")
def count_up_to(max_value):
    count = 1
    while count <= max_value:
        yield count
        count += 1

for num in count_up_to(3):
    print(num)

print("\n--- Generator expression ---")
for num in (i for i in range(1, 3 + 1)):
    print(num)

--- Generator function ---
1
2
3

--- Generator expression ---
1
2
3


# Concurrency

Python mainly has three built-in modules to implement concurrency: `asyncio`, `threading`, and `multiprocessing`.

## asyncio

`asyncio` lets you achieve concurrency via an event loop - a single-threaded, cooporative scheduler.

The event loop acts as the almighty scheduler. It has a *ready queue*, which holds callbacks that are ready to run, plus some extra stuff like timers and I/O multiplexer (don't know too much about this). Essentially, what it does is it pops the callbacks from the ready queue and executes them until the queue is empty. A technical note here is that the callbacks are `asyncio.Task` objects wrapped in `asyncio.Handle` objects - the reason this is necessary is because an event loop is fundamentally a callback dispatcher, meaning it only knows how to run callbacks. So, at a high level, `asyncio.Task` objects can be thought of as the fundamental unit of scheduling in `asyncio`.

Now let's walk through how an event loop works. 

First, you initialise an event loop via `asyncio.run(main())`. This is a very nice abstraction over initialising, queueing the main coroutine, and cleaning up.

Next, you schedule coroutines via `asyncio.create_task(coroutine())`. An important detail when you queue a coroutine is that it wraps the coroutine object in a `asyncio.Task` object. Because `Task` inherits from `Future`, it can track its state (pending, cancelled, or finished) which allows the event loop to interleave tasks. 

Python allows you to await any object whose type implements the `__await__` method. This is the awaitable protocol. The behaviour depends on how the `__await__` method is implemented, for example,

* Awaiting a plain coroutine returns an iterator which the interpreter steps through until it raises a `StopIteration` exception. This means the coroutines are just delegating to one another, essentially executing synchronously on a nested call stack.
* Awaiting a Future/Task, however, yields itself if it isn't finished. This propogates (bubbles) all the way back to the event loop, which is how it explicitly yields control back to the event loop. If the Future/Task is finished, it grabs the result and continues.

A takeaway here is that in `asyncio`, the fundamental unit of concurrency is a `Task` object, not a `Coroutine` object.

In [7]:
import asyncio


async def plain_inner():
    print("      -> [Plain Inner] started executing")
    print("      -> [Plain Inner] finished")


async def plain_outer(name):
    print(f"  [Plain Outer {name}]: started")
    # Awaiting another plain coroutine directly creates a nested call stack
    await inner_plain()
    print(f"  [Plain Outer {name}]: finished")


async def worker(name, delay):
    print(f"Task {name}: started, sleeping for {delay}s")
    # Awaiting a Future/Task yields control back to the event loop
    await asyncio.sleep(delay)
    print(f"Task {name}: resumed and finished!")


async def main():
    print("--- Event loop starts ---")

    print("\n--- Schedule tasks ---")
    task1 = asyncio.create_task(worker("A", 1.0))
    task2 = asyncio.create_task(worker("B", 0.5))
    
    print("\n--- Await plain coroutines (synchronous) ---")
    await outer_plain("X")
    await outer_plain("Y")
    
    print("\n--- Await tasks (concurrent) ---")    
    await task1
    await task2
    
    print("\n--- Event loop ends ---")


# Run this in your Jupyter cell:
await main()

--- Event loop starts ---

--- Schedule tasks ---

--- Await plain coroutines (synchronous) ---
  [Outer Plain X]: started
      -> [Inner Plain] started executing
      -> [Inner Plain] finished
  [Outer Plain X]: finished
  [Outer Plain Y]: started
      -> [Inner Plain] started executing
      -> [Inner Plain] finished
  [Outer Plain Y]: finished

--- Await tasks (concurrent) ---
Task A: started, sleeping for 1.0s
Task B: started, sleeping for 0.5s
Task B: resumed and finished!
Task A: resumed and finished!

--- Event loop ends ---


A breakdown of what happens after both Task A and Task B yields.

1. Since the ready queue is empty, the event loop looks at its internal timers (where asyncio.sleep keeps track of deadlines).

2. Task B was sleeping for 0.5 seconds, so its timer expires first. The event loop moves Task B's continuation callback back into the ready queue.

3. The event loop pops Task B from the ready queue, resumes it, it prints "Task B: resumed and finished!", and then Task B exits.

4. Time ticks forward to 1.0 seconds. Task A's timer expires, and the event loop moves Task A's callback into the ready queue.

5. The event loop pops Task A, resumes it, it prints "Task A: resumed and finished!", and Task A exits.

6. Because main was doing await task1, and task1 is now finished, main's future is satisfied. main is pushed back into the ready queue, wakes up, proceeds to await task2 (which is already done), and finishes the program.

> TODO: Learn what happens when tasks finish (timer, network request, awaiting on other tasks)

## threading


The `threading` module lets you spawn multiple operating system threads. However, because of the GIL, Python only allows one thread to be executed at a time.

## multiprocessing

The `multiprocessing` module lets you spawn separate operating system processes, each with its own memory and Python interpreter. This means you can by pass the GIL and run tasks on multiple cores, which gives true parallelism. The catch is that the processes don't share memory so communication adds complexity and there is a significantly larger overhead.

## trade-offs

Both `threading` and `asyncio` are good for I/O bound tasks. A key difference is that `asyncio` uses coorporate scheduling, where `async` tasks (coroutines) explicitly yield control back to the event loop whenever there is an `await` statement, whereas `threading` uses premptive scheduling, where the OS forcefully interupts threads and switches to another. Generally, `asyncio` is preferred because the the event loop has very small context-switching overhead and can scale a lot better. However, one disadvantage is that it cannot natively handle blaocking code like synchronous tasks or CPU-heavy tasks. Multithreading can integrate easily with legacy synchronous codebases since it gives every thread a fair slice of CPU time. However, its disadvantage is that it has slightly more overhead and can create race conditions so there is an extra complexity from managing synchronisation primitives.

# Syntax

## Native syntax

These are Python's fundamental primitives

* `def` / `class`
* `if` / `elif` / `else`
* `while`
* `try` / `except` / `finally`
* etc...

## Syntactic sugar

These are features that makes code cleaner. Under the hood, Python translates it into the primitives above

* Operators (`+`, `-`, `*`, `==`), which maps to dunder methods like `__add__`, `__sub__`, `__mul__`, and `__eq__`
* Method calls, which maps `obj.method(10)` to `type(obj).method(obj, 10)`
* Decorators, which is a shorthand for `func = my_decorator(func)`
* `for` loops, which translates the below

In [13]:
my_list = [1, 2, 3]


print("--- For loop ---")
for item in my_list:
    print(item)


print("--- Under the hood ---")
iterator = iter(my_list)

try:
    while True:
        item = next(iterator)
        print(item)
except StopIteration:
    pass

--- For loop ---
1
2
3
--- Under the hood ---
1
2
3


# Duck typing

This is the classic "if it walks like a duck and quacks like a duck, it's a duck"

That is, instead of asking "are you an instance of `Duck`?", Python asks "can you perform the actions a `Duck` performs?'

For example, iterable objects (objects that can be looped over in a for loop) don't need to inherit from a special class, they just have to implement the `__iter__` method, which returns an `iterator` object.

Similarly, callable objects (objects that can be invoked with parentheses) don't need to inherit from a special class, they  just have to implement the `__call__` method.

Similarly, awaitable objects (objects that can be awaited in `asyncio`) don't need to inherit from a special class, they just have to implement the `__await__` method.